<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week9_instructor_power_calc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 – Instructor demo: power calculation, live

**For live use in the workshop, not a student handout.** Reproduces the slide's own power-calculation code (`NormalIndPower`), answers the Task's "what if the effect is smaller?" question, and adds a reusable function so you can answer a spontaneous "what about X%?" question from the room without leaving the notebook.

## Setup

In [ ]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower
import numpy as np
import matplotlib.pyplot as plt

def n_per_arm(r0, r1, alpha=0.05, power=0.8):
    """Required sample size per arm to detect a change from r0 to r1 in a proportion (two-sample test)."""
    effect_size = proportion_effectsize(r0, r1)
    return NormalIndPower().solve_power(effect_size=abs(effect_size), alpha=alpha, power=power, ratio=1)

## The slide's own scenario: detecting a doubling (1% -> 2%)

In [ ]:
n_doubling = n_per_arm(0.01, 0.02)
print(f'To detect 1% -> 2% (doubling) at 80% power: {n_doubling:,.0f} visitors per arm')

This should match the number already shown on the slide – run it to confirm before moving on.

## The Task: what if you only care about a 50% relative lift (1% -> 1.5%)?

In [ ]:
n_modest = n_per_arm(0.01, 0.015)
print(f'To detect 1% -> 1.5% (a 50% relative lift) at 80% power: {n_modest:,.0f} visitors per arm')
print(f'That is {n_modest / n_doubling:.1f}x more data than the doubling scenario needed.')

**Talking point:** roughly 3.4x more data for a more modest, but still commercially meaningful, effect. This is the "smaller effects need bigger samples" principle from the slide, made concrete – precision doesn't come free, and it doesn't scale linearly with how much smaller the effect you're chasing is.

## Answering a live "what if" question

Use `n_per_arm(r0, r1)` for any baseline/target the class asks about. For example, if someone asks "what if we only needed to detect a 25% relative lift?":

In [ ]:
n_per_arm(0.01, 0.0125)  # a 25% relative lift: 1% -> 1.25%

## Optional: how required sample size scales with effect size

In [ ]:
relative_lifts = np.array([0.10, 0.25, 0.50, 0.75, 1.00])  # 10% to 100% (doubling) relative lift
baseline = 0.01
required_n = [n_per_arm(baseline, baseline * (1 + lift)) for lift in relative_lifts]

plt.figure(figsize=(6, 4))
plt.plot(relative_lifts * 100, required_n, marker='o', color='#457b9d')
plt.xlabel('Relative lift being detected (%)')
plt.ylabel('Required n per arm')
plt.title('Smaller effects need (much) bigger samples')
plt.show()